In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.utils import resample

nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#Load spaCy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [ ]:
#Load Dataset
df = pd.read_csv("twitter_sentiment_data.csv")

df = df.rename(columns={"message": "text"})
df = df[['text', 'sentiment']].dropna()

In [ ]:
#Convert Sentiment Labels
df = df[df['sentiment'] != 0]
print (df)

                                                    text  sentiment
0      @tiniebeany climate change is an interesting h...         -1
1      RT @NatGeoChannel: Watch #BeforeTheFlood right...          1
2      Fabulous! Leonardo #DiCaprio's film on #climat...          1
3      RT @Mick_Fanning: Just watched this amazing do...          1
4      RT @cnalive: Pranita Biswasi, a Lutheran from ...          2
...                                                  ...        ...
43937  RT @cnni: Leonardo DiCaprio: $q$Not one questi...          2
43938  Dear @realDonaldTrump,\nYeah right. Human Medi...          1
43939  What will your respective parties do to preven...          1
43940  RT @MikkiL: UN Poll Shows Climate Change Is th...          2
43942  @Likeabat77 @zachhaller \n\nThe wealthy + foss...          1

[36228 rows x 2 columns]


In [ ]:
df['label'] = df['sentiment'].map({
    1: "positive",
    -1: "negative"
})

In [ ]:
df_pos = df[df.label == "positive"]
df_neg = df[df.label == "negative"]

max_size = min(len(df_pos), len(df_neg))

df_pos = df_pos.sample(max_size, random_state=42)
df_neg = df_neg.sample(max_size, random_state=42)

df = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42)
print (df)

                                                    text  sentiment     label
43116  #NoDakotaAccess  #wATERiSlIFE #sacredsites htt...          1  positive
444    RT @alicebell: We don't need a 'warÃ¢â‚¬â„¢ on...          1  positive
39326  RT @ShujaRabbani: Prince Charles said WHAT?!! ...         -1  negative
7080   @TedAbram1 @wattsupwiththat you kooks...climat...         -1  negative
40798  All those idiots that believe in that climate ...         -1  negative
...                                                  ...        ...       ...
8430   @PatriotByGod In Laos on his final Asian tour ...         -1  negative
21481  RT @Doc_0: The Great Anti-Trump Bellwether Ele...         -1  negative
19633  I helped fight climate change by donating to o...          1  positive
2705   All American climate change theorists sitting ...         -1  negative
39178  RT @zerohedge: Another thriving market for Gol...         -1  negative

[7980 rows x 3 columns]


In [ ]:
#FIX STOPWORDS
stop_words = set(stopwords.words('english'))

# Keep important negation words
negations = {"not", "no", "nor", "n't"}
stop_words = stop_words - negations
print (stop_words)

{"we're", 'couldn', 'o', 'm', 'after', 'mightn', 'and', 'same', 'between', 'was', "she'll", "aren't", "he'd", 'him', "isn't", 'but', 'it', 'because', "i'll", "they'll", 'your', 'herself', "mustn't", 'having', "we've", 'by', "wasn't", 'can', 'over', 'd', 'down', 'her', 'an', 'his', 'aren', "didn't", 'y', 'again', 'myself', 'now', "doesn't", "she's", 'off', 'those', 'are', 'does', 'should', 'if', 'been', 'as', "should've", 'there', 'them', "i'd", 'above', 'under', 'being', 'haven', 'during', "you're", 'shan', 'with', 'doing', 'won', "it'll", "that'll", 'all', 'mustn', 'very', 'needn', 're', 'have', 'itself', "hasn't", 'wouldn', "it's", "shouldn't", 'up', 'against', 'why', 'don', 'only', 'shouldn', "he'll", 'am', 'isn', 'has', 'hasn', 'until', 'these', 'me', 'some', 'when', 'ourselves', 'their', 'doesn', 'they', "he's", "i've", 'where', 'is', 'for', 'most', "you'd", 'had', 'both', 'my', 'yourselves', "she'd", 'hers', "it'd", 'few', 's', "won't", "don't", 'how', 'this', "shan't", "haven't"

In [ ]:
#CLEANING + ASPECT EXTRACTION
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]

    return " ".join(words)

df['clean_text'] = df['text'].apply(clean)

In [ ]:
X = df['clean_text']
y = df['label']

print (X,y)

43116               nodakotaaccess waterislife sacredsites
444      rt alicebell dont need war climate change need...
39326    rt shujarabbani prince charles said maybe davi...
7080     tedabram wattsupwiththat kooksclimate change s...
40798    idiots believe climate change crap u come shov...
                               ...                        
8430     patriotbygod laos final asian tour climate cha...
21481    rt doc great antitrump bellwether election lik...
19633    helped fight climate change donating offset po...
2705     american climate change theorists sitting marr...
39178    rt zerohedge thriving market goldmanqs carbon ...
Name: clean_text, Length: 7980, dtype: object 43116    positive
444      positive
39326    negative
7080     negative
40798    negative
           ...   
8430     negative
21481    negative
19633    positive
2705     negative
39178    negative
Name: label, Length: 7980, dtype: object


In [ ]:
#Vectorization TF-IDF
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=3
)

X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

print(X,y)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 99414 stored elements and shape (7980, 6563)>
  Coords	Values
  (1, 4759)	0.07631513443121904
  (1, 1888)	0.21756715065182372
  (1, 3835)	0.4887497853118187
  (1, 6125)	0.30299113682807255
  (1, 1248)	0.06771578534799179
  (1, 768)	0.06882225365012286
  (1, 4716)	0.4246807345877648
  (1, 1900)	0.40465483162470767
  (1, 6126)	0.3747178122807504
  (1, 1256)	0.06947777999807556
  (1, 987)	0.34278092646957614
  (2, 4759)	0.06529198634888644
  (2, 4359)	0.726677583208091
  (2, 1153)	0.3633387916040455
  (2, 5024)	0.23177425720514164
  (2, 3618)	0.25922634241628617
  (2, 5397)	0.28762630044446275
  (2, 6487)	0.3633387916040455
  (3, 768)	0.0788643800426551
  (3, 6328)	0.429394365762805
  (3, 2816)	0.2466027456468312
  (3, 2587)	0.29321573955059743
  (3, 4513)	0.4157454044429547
  (3, 5627)	0.3655270278566973
  (3, 2570)	0.24382200072041976
  :	:
  (7976, 2240)	0.3558054963650437
  (7976, 1845)	0.3644550657226053
  (7976, 242)	0.40

In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC()
model.fit(X, y)

LinearSVC()

In [ ]:
#TRAIN MODEL
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
#Evaluate
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

# Re-train the model to match the current feature dimension of X_train (8000 features)
model = LinearSVC()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.85      0.81      0.83       798
    positive       0.82      0.86      0.84       798

    accuracy                           0.84      1596
   macro avg       0.84      0.84      0.84      1596
weighted avg       0.84      0.84      0.84      1596



In [ ]:
def predict(text):
    cleaned = clean(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]

    print("\nText:", text)
    print("Sentiment:", pred)

predict("The weather is beautiful today, let's go for outing.")
predict("Global warming is becoming harmful day by day")


Text: The weather is beautiful today, let's go for outing.
Sentiment: positive

Text: Global warming is becoming harmful day by day
Sentiment: negative
